In [0]:
%pip install groq

In [0]:
%restart_python

In [0]:
# %run ./tools/registry

In [0]:
%run ./utils/policy_extractor

In [0]:
%run ./utils/chat_history


In [0]:
# tools = [
#     ### Ascertain the policy in question
#     {
#         "type": "function",
#         "function": {
#             "name": "extract_policies",
#             "description": "Search for all the possible policy numbers related to the customer based on available information if it has not been ascertained",
#             "parameters": {
#                 "type": "object",
#                 "properties": {
#                     "chat_history": {"type": "string", "description": "Chat history"}
#                 },
#                 "required": ["chat_history"],
#             },
#         },
#     },
#     ### Get policy summary

#     ### Get debit order info

#     ### Send document

# ]

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string", "description": "City and state, e.g. San Francisco, CA"}
                },
                "required": ["location"],
            },
        },
    }
]

In [0]:
base_prompt = """# CONTEXT
You are an insurance customer service assistant for ABSA bank in South Africa. 

## RULES AND GUIDELINES
### You must:
- act as if you were an employee of ABSA - governed by ABSA's code of conduct
- respond in a friendly and professional manner

### You CANNOT:
- offer any advise as you are not a licenced financial advisor
- say anything that will harm ABSA's reputation

### What you can assist with
You CAN assist customers with the below:
- Answer policy related information:
    - Insured items on the current policy
    - Debit order information
    - Policy summary
    - Terms and conditions

### What to do if the request falls outside of the domain of your expertise
If the customer is asking about something you cannot assist with, ask if you can transfer them to a different agent.

### What to do if the customer is not clear about their request
If the customer is not clear about their request, ask them to rephrase their request.

## CUSTOMER INFORMATION (POSSIBLY RELATED POLICIES)
{policy_summary}

**NOTE** - It is possible for the customer to have multiple policies with ABSA and one / some of them are not listed above. If you are not sure which policy the customer is referring to, ask the customer to confirm which policy they are referring to.

"""

In [0]:
class Agent:

    def __init__(self):
        from groq import Groq
        self.client = Groq(api_key="")

    def get_resp(self, messages: List[Dict[str, str]], tools = None):
    
        if tools:
            completion = self.client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=messages,
                temperature=0,
                max_completion_tokens=2048,
                top_p=1,
                stream=False,
                stop=None,
                tools=tools
            )

        else:
            completion = self.client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=messages,
                temperature=0,
                max_completion_tokens=2048,
                top_p=1,
                stream=False,
                stop=None,
            )

        return completion.choices[0]

In [0]:
import copy
from pprint import pprint
from typing import Dict, Any, List, Optional, Tuple

class CustomerServiceAgent:

    def __init__(self):
        # self.registry = tool_registry
        self.history = chat_history
        self.agent = Agent()

    def process_message(
        self, 
        message: str, 
        session_id: Optional[str] = None
    ) -> Dict[str, Any]:

        # Get required history and state
        chat_history = self.history.get_history(session_id)
        state = self.history.get_session_state(session_id) or {}
        customer_identifier = state.get("policy_number", None)

        # Construct dummy chat history
        message_clean = message.strip()

        if chat_history:
            dummy_history = copy.deepcopy(chat_history)
        else:
            dummy_history = [{'role': 'user', 'content': message_clean}]

        # Call possible policy identifier everytime
        possible_policy_numbers = policy_extractor.extract_policies(str(dummy_history))

        # Get policy info for system prompt
        policy_summary = policy_extractor.get_summary(possible_policy_numbers)
        
        system_prompt = [{
                "role": "system",
                "content": base_prompt.format(policy_summary=policy_summary)
            }]

        # Insert into dummy history
        dummy_history = system_prompt + dummy_history

        # return dummy_history

        while True:
            # Pass to LLM
            resp = self.agent.get_resp(
                dummy_history,
                tools
                )
            print(resp)

            # If tool selected
            if resp.message.tool_calls:
            # dummy_history
            #   execute tool(s)
            #   append to history
              continue
            else:
              break

        # # Construct final response 
        agent_response = resp.message.content

        # 5. Record turn in session history
        self.history.record_turn(
            session_id, message_clean, agent_response,
            turn_metadata={
                # "intent": tool_name,
                # "policy_number": policy_number,
                # "tool_result_status": tool_result.get("status"),
            }
        )

        # return {
        #     "status": "SUCCESS",
        #     "session_id": session_id,
        #     "policy_number": policy_number,
        #     "intent_detected": tool_name,
        #     "tool_called": tool_name,
        #     "tool_parameters": tool_kwargs,
        #     "tool_result": tool_result,
        #     "agent_response": agent_response,
        #     "conversation_history": self.history.get_history(session_id) if session_id else []
        # }


# Global Agent Instance
agent = CustomerServiceAgent()


In [0]:
# from typing import Dict, Any, List, Optional, Tuple

# class CustomerServiceAgent:

#     def __init__(self):
#         self.registry = tool_registry
#         self.history = chat_history

#     def process_message_llm(
#         self, 
#         message: str, 
#         session_id: Optional[str] = None
#     ) -> Dict[str, Any]:

#         # Get required history and state
#         chat_history = self.history.get_history(session_id)
#         state = self.history.get_session_state(session_id)
#         customer_identifier = state.get("policy_number", None)

#         message_clean = message.strip()

#         # If no customer identifier, try to extract from message
#         if not customer_identifier:
#             if chat_history:
#                 chat_history_str = "\n".join([f"{turn['role']}: {turn['content']}" for turn in chat_history])
#                 chat_history_str += "\n" + f"user: {message_clean}"
#             else:
#                 chat_history_str = message_clean

#             possible_customer_identifier = policy_extractor.extract_policies(chat_history_str)

#             if possible_customer_identifier and len(possible_customer_identifier) == 1:
#                 self.history.update_session_state(session_id, possible_customer_identifier[0])
#                 customer_identifier = possible_customer_identifier[0]

#         # 1. Disambiguate policy context
#         if customer_identifier:
#             policy_context = {
#                 "status": "SINGLE_POLICY_FOUND",
#                 "policy_number": customer_identifier,
#                 "needs_disambiguation": False
#             }
#         else:
#             policy_context = policy_extractor.resolve_policy_context(message_clean, possible_customer_identifier)

#         # If policy is ambiguous or missing, return prompt to user
#         if policy_context["needs_disambiguation"]:
#             response = policy_context["disambiguation_prompt"]
#             if session_id:
#                 self.history.record_turn(
#                     session_id, message_clean, response,
#                     turn_metadata={"status": "DISAMBIGUATION_REQUIRED"}
#                 )
#             return {
#                 "status": "DISAMBIGUATION_REQUIRED",
#                 "session_id": session_id,
#                 "policy_context": policy_context,
#                 "agent_response": response,
#                 "tool_called": None,
#                 "tool_result": None,
#                 "conversation_history": self.history.get_history(session_id) if session_id else []
#             }

#         policy_number = policy_context["policy_number"]

#         # 2. Intent Classification and Tool Routing
#         tool_name, tool_kwargs = self._select_tool_and_args(message_clean, policy_number)

#         if not tool_name:
#             # Fallback to general policy summary if intent unrecognized
#             tool_name = "get_policy_summary"
#             tool_kwargs = {"policy_number": policy_number}

#         # 3. Execute Tool via Tool Registry
#         tool_result = self.registry.execute(tool_name, **tool_kwargs)

#         # 4. Synthesize final response
#         agent_response = self._synthesize_response(message_clean, tool_name, tool_result)

#         # 5. Record turn in session history
#         if session_id:
#             self.history.record_turn(
#                 session_id, message_clean, agent_response,
#                 turn_metadata={
#                     "intent": tool_name,
#                     "policy_number": policy_number,
#                     "tool_result_status": tool_result.get("status"),
#                 }
#             )

#         return {
#             "status": "SUCCESS",
#             "session_id": session_id,
#             "policy_number": policy_number,
#             "intent_detected": tool_name,
#             "tool_called": tool_name,
#             "tool_parameters": tool_kwargs,
#             "tool_result": tool_result,
#             "agent_response": agent_response,
#             "conversation_history": self.history.get_history(session_id) if session_id else []
#         }

#     def process_message(
#         self, 
#         message: str, 
#         session_id: Optional[str] = None
#     ) -> Dict[str, Any]:

#         # Get required history and state
#         chat_history = self.history.get_history(session_id)
#         state = self.history.get_session_state(session_id)
#         customer_identifier = state.get("policy_number", None)

#         message_clean = message.strip()

#         # If no customer identifier, try to extract from message
#         if not customer_identifier:
#             if chat_history:
#                 chat_history_str = "\n".join([f"{turn['role']}: {turn['content']}" for turn in chat_history])
#                 chat_history_str += "\n" + f"user: {message_clean}"
#             else:
#                 chat_history_str = message_clean

#             possible_customer_identifier = policy_extractor.extract_policies(chat_history_str)

#             if possible_customer_identifier and len(possible_customer_identifier) == 1:
#                 self.history.update_session_state(session_id, possible_customer_identifier[0])
#                 customer_identifier = possible_customer_identifier[0]

#         # 1. Disambiguate policy context
#         if customer_identifier:
#             policy_context = {
#                 "status": "SINGLE_POLICY_FOUND",
#                 "policy_number": customer_identifier,
#                 "needs_disambiguation": False
#             }
#         else:
#             policy_context = policy_extractor.resolve_policy_context(message_clean, possible_customer_identifier)

#         # If policy is ambiguous or missing, return prompt to user
#         if policy_context["needs_disambiguation"]:
#             response = policy_context["disambiguation_prompt"]
#             if session_id:
#                 self.history.record_turn(
#                     session_id, message_clean, response,
#                     turn_metadata={"status": "DISAMBIGUATION_REQUIRED"}
#                 )
#             return {
#                 "status": "DISAMBIGUATION_REQUIRED",
#                 "session_id": session_id,
#                 "policy_context": policy_context,
#                 "agent_response": response,
#                 "tool_called": None,
#                 "tool_result": None,
#                 "conversation_history": self.history.get_history(session_id) if session_id else []
#             }

#         policy_number = policy_context["policy_number"]

#         # 2. Intent Classification and Tool Routing
#         tool_name, tool_kwargs = self._select_tool_and_args(message_clean, policy_number)

#         if not tool_name:
#             # Fallback to general policy summary if intent unrecognized
#             tool_name = "get_policy_summary"
#             tool_kwargs = {"policy_number": policy_number}

#         # 3. Execute Tool via Tool Registry
#         tool_result = self.registry.execute(tool_name, **tool_kwargs)

#         # 4. Synthesize final response
#         agent_response = self._synthesize_response(message_clean, tool_name, tool_result)

#         # 5. Record turn in session history
#         if session_id:
#             self.history.record_turn(
#                 session_id, message_clean, agent_response,
#                 turn_metadata={
#                     "intent": tool_name,
#                     "policy_number": policy_number,
#                     "tool_result_status": tool_result.get("status"),
#                 }
#             )

#         return {
#             "status": "SUCCESS",
#             "session_id": session_id,
#             "policy_number": policy_number,
#             "intent_detected": tool_name,
#             "tool_called": tool_name,
#             "tool_parameters": tool_kwargs,
#             "tool_result": tool_result,
#             "agent_response": agent_response,
#             "conversation_history": self.history.get_history(session_id) if session_id else []
#         }

#     def _select_tool_and_args(self, message: str, policy_number: str) -> Tuple[Optional[str], Dict[str, Any]]:

#         msg_lower = message.lower()

#         # Intent: Vehicle Noting of Interest
#         if any(kw in msg_lower for kw in ["noting of interest", "noted interest", "bank confirmation", "vehicle noting", "financier"]):
#             # Extract potential bank name or vehicle reg
#             bank = None
#             for b in ["standard bank", "absa", "nedbank", "fnb", "wesbank"]:
#                 if b in msg_lower:
#                     bank = b.title()
#                     break
#             return "send_vehicle_noting_of_interest", {
#                 "policy_number": policy_number,
#                 "financial_institution": bank
#             }

#         # Intent: Policy Schedule Document
#         elif any(kw in msg_lower for kw in ["policy schedule", "schedule document", "send schedule", "copy of policy", "policy document"]):
#             return "send_policy_schedule", {
#                 "policy_number": policy_number
#             }

#         # Intent: Insurance Certificate / Tax / Border
#         elif any(kw in msg_lower for kw in ["certificate", "confirmation of cover", "border cross", "tax invoice"]):
#             cert_type = "CONFIRMATION_OF_COVER"
#             if "border" in msg_lower:
#                 cert_type = "BORDER_CROSSING"
#             elif "tax" in msg_lower:
#                 cert_type = "TAX_INVOICE"
#             return "send_insurance_certificate", {
#                 "policy_number": policy_number,
#                 "certificate_type": cert_type
#             }

#         # Intent: Debit Order & Billing Questions
#         elif any(kw in msg_lower for kw in ["debit order", "debit date", "billing", "bank account", "payment date", "debit amount", "how much is my debit"]):
#             return "get_debit_order_info", {
#                 "policy_number": policy_number
#             }

#         # Intent: Payment History
#         elif any(kw in msg_lower for kw in ["payment history", "past payments", "ledger", "statement"]):
#             return "get_payment_history", {
#                 "policy_number": policy_number,
#                 "num_months": 6
#             }

#         # Intent: Policy Terms and Conditions / Clause Q&A
#         elif any(kw in msg_lower for kw in ["terms", "conditions", "excess", "exclusion", "covered", "clause", "what is covered", "rules"]):
#             return "search_policy_terms_and_conditions", {
#                 "policy_number": policy_number,
#                 "query": message
#             }

#         # Intent: Policy Overview / Details
#         elif any(kw in msg_lower for kw in ["policy details", "status", "summary", "overview"]):
#             return "get_policy_summary", {
#                 "policy_number": policy_number
#             }

#         return None, {}

#     def _synthesize_response(self, user_message: str, tool_name: str, tool_result: Dict[str, Any]) -> str:
#         """Format final customer response based on tool execution result."""
#         if tool_result.get("status") == "ERROR":
#             return f"I ran into an issue while processing your request: {tool_result.get('error')}"

#         data = tool_result.get("data", {})

#         if tool_name == "send_policy_schedule":
#             return (
#                 f"I have successfully generated and dispatched your Policy Schedule for **{data['policy_number']}**.\n\n"
#                 f"• **Destination**: `{data['recipient_email']}`\n"
#                 f"• **File Name**: `{data['file_name']}`\n"
#                 f"• **Download Link**: {data['download_url']}"
#             )

#         elif tool_name == "send_vehicle_noting_of_interest":
#             return (
#                 f"Your Vehicle Noting of Interest confirmation certificate has been issued for **{data['policy_number']}**.\n\n"
#                 f"• **Vehicle**: {data['vehicle']}\n"
#                 f"• **Financial Institution**: {data['financial_institution']}\n"
#                 f"• **Sent To**: `{data['recipient_email']}`\n"
#                 f"• **Download Certificate**: {data['download_url']}"
#             )

#         elif tool_name == "get_debit_order_info":
#             return (
#                 f"Here are the debit order details for policy **{data['policy_number']}** ({data['product_type']}):\n\n"
#                 f"• **Debit Day**: {data['debit_day']}\n"
#                 f"• **Debit Amount**: {data['debit_amount']}\n"
#                 f"• **Bank & Account**: {data['bank_name']} ({data['account_number_masked']})\n"
#                 f"• **Last Debit Date**: {data['last_successful_debit_date']} ({data['last_debit_status']})\n"
#                 f"• **Next Debit Date**: {data['next_debit_date']}"
#             )

#         elif tool_name == "search_policy_terms_and_conditions":
#             clauses = data.get("relevant_terms_and_conditions", [])
#             clauses_text = "\n\n".join([f"**[{c['section']}]**\n_{c['text']}_" for c in clauses])
#             return (
#                 f"Here is what your policy terms and conditions state regarding **'{data['query']}'** for **{data['policy_number']}** ({data['policy_type']}):\n\n"
#                 f"{clauses_text}"
#             )

#         elif tool_name == "get_policy_summary":
#             return (
#                 f"Policy Summary for **{data['policy_number']}** ({data['product_type']}):\n\n"
#                 f"• **Status**: {data['status']}\n"
#                 f"• **Premium**: {data['premium_amount']} per month\n"
#                 f"• **Basic Excess**: {data['basic_excess']}\n"
#                 f"• **Inception Date**: {data['start_date']}\n"
#                 f"• **Cover Details**: {data['cover_details']}"
#             )

#         return data.get("message", "Your request has been processed successfully.")

# # Global Agent Instance
# agent = CustomerServiceAgent()


In [0]:
resp = agent.process_message(message = 'Hello. What policies do I currently have? john.smith@email.com', session_id = "1234")
pprint(resp)

In [0]:
resp[0].message.tool_calls

In [0]:
# agent.process_message(message = 'POL100001', session_id = "123")

In [0]:
# agent.history.get_session_state("123")
# agent.history.get_history("123")